# LangGraph G7 — Knowledge and retrieval
CampusAI's `search_handbook` matches words. Ask *"How long can I keep a book?"* and it finds
nothing, because the library note says "borrowed for 21 days". Real knowledge access needs
**retrieval by meaning**:

```text
Indexing (once)                          Query time (every question)
Documents  -> chunks -> embeddings         question -> embedding -> nearest chunks -> model + chunks -> grounded answer
                        (vectors)                                   (cosine similarity)
```

An **embedding model** turns text into a vector so that similar meanings land near each other.
We build the whole pipeline from first principles in a few lines of numpy, then use it two ways:
as a **retrieval node** in the faq desk (fixed, predictable: "2-step RAG"), and as a **tool** the
records agent can call when it decides it needs policy ("agentic RAG"). Either way, the prompt
tells the model to answer **only from the excerpts** and to say when they do not contain the
answer: that instruction is what makes the answer *grounded*.

In [ ]:
%pip install -q -U sentence-transformers

### Step 1 — Chunks, embeddings and cosine search in plain numpy

The embedding model runs locally (about 90 MB, downloaded once). If it cannot be loaded, a
keyword embedding keeps the section runnable, with weaker matching.

In [ ]:
import numpy as np                                                  # numpy

KNOWLEDGE = [{"source": f"handbook/{topic}", "text": text} for topic, text in HANDBOOK.items()]   # ours: the corpus
KNOWLEDGE += [{"source": f"faq/{topic}", "text": text} for topic, text in FAQ_DOCS.items()]
KNOWLEDGE += [{"source": f"catalogue/{code}", "text": f"{code} {c['title']}: {c['description']} {c['credits']} credits."} for code, c in COURSES.items()]

class KeywordEmbedder:                                              # ours: fallback, a hashed bag of words
    def encode(self, texts):
        vectors = np.zeros((len(texts), 256))
        for i, text in enumerate(texts):
            for word in re.findall(r"[a-z]+", text.lower()):
                vectors[i, hash(word) % 256] += 1.0
        return vectors

try:
    from sentence_transformers import SentenceTransformer           # sentence-transformers: local embedding models
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    print("embeddings : all-MiniLM-L6-v2 (semantic)")
except Exception as exc:
    embedder = KeywordEmbedder()
    print("embeddings : keyword fallback (", type(exc).__name__, ")")

def embed(texts):                                                   # ours: texts -> unit-length vectors
    vectors = np.asarray(embedder.encode(list(texts)), dtype=float)
    return vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9)

INDEX = embed(doc["text"] for doc in KNOWLEDGE)                     # ours: one vector per chunk, computed once
print("index      :", INDEX.shape, "= chunks x dimensions")

def retrieve(query, k=3):                                           # ours: cosine similarity = dot product of unit vectors
    scores = INDEX @ embed([query])[0]
    best = np.argsort(-scores)[:k]
    return [{"score": float(scores[i]), **KNOWLEDGE[i]} for i in best]

for hit in retrieve("How long can I keep a book?"):
    print(f"  {hit['score']:.2f} [{hit['source']}] {hit['text'][:70]}")

### Step 2 — A retrieval node for the faq desk, and a retrieval tool for the agent

The faq desk becomes retrieve-then-answer with no loop. The records agent gets `search_knowledge`
and decides for itself when to call it. `build_desks()` from G4 accepts both replacements.

In [ ]:
def faq_rag(state: DeskState):                                      # ours: retrieval node + grounded answer node, in one
    question = text_of(state["messages"][-1])
    hits = retrieve(question, k=3)
    excerpts = "\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    reply = model.invoke([SystemMessage("Answer only from the excerpts below and cite the source in brackets. If they do not contain the answer, say so.\n\n" + excerpts), HumanMessage(question)])   # LangChain
    return {"messages": [reply]}

@tool
def search_knowledge(query: str) -> str:
    """Search the handbook, FAQ notes and course catalogue by meaning. Returns the three most relevant excerpts with sources."""
    return "\n".join(f"[{h['source']}] {h['text']}" for h in retrieve(query, k=3))

KNOWLEDGE_TOOLS = [get_student, get_course, search_knowledge]      # ours: search_knowledge replaces search_handbook from here on
records_rag_agent = build_agent(KNOWLEDGE_TOOLS)
campusai_v7 = build_desks(faq_node=faq_rag, records_node=records_rag_agent)

for q in ["How long can I keep a library book?", "Student S001 has 68% attendance; can they sit the CS201 exam?"]:
    out = campusai_v7.invoke({"messages": [HumanMessage(q)]})
    print(f"\n[{out['category']}] {q}")
    show_messages(out["messages"][1:])

### Recap

- **Problem seen:** keyword search could not find text that meant the same thing in different words.
- **Layer added:** an embedding index with cosine search, a retrieval node for the fixed desk and a retrieval tool for the agent, both with grounding instructions.
- **Evidence:** the library note was found for a question that shared no words with it; the agent combined a record with a policy excerpt.